# DSCI 351 — Project Part 2: PySpark Analysis

**Netflix Analysis Part 2** — PySpark queries, analytics, and performance optimization on the raw Netflix catalog.

**Sections:**
- **Part 2.1** — Recreate Part 1 SQL questions Q1–Q8 in PySpark
- **Part 2.2.1** — Deeper analytics: genre breadth, catalog depth, rating vs breadth, added-year lag
- **Part 2.2.2** — Query optimization on the lag summary (baseline / `.explain(True)` / cache / column pruning / repartition)

Runs on Google Colab out of the box.


## Setup

Install PySpark and load the raw uncleaned `netflix_titles.csv` before merging.

In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as fc

spark = SparkSession.builder.appName("dsci351-netflix-part2").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)

In [ ]:
# Upload netflix_titles.csv (raw, uncleaned) and movies.csv when prompted.
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load the raw netflix catalog and the small movie lookup table exported from MySQL.
netflix = spark.read.csv("netflix_titles.csv", header=True, inferSchema=True)
movie_df = spark.read.csv("movies.csv",        header=True, inferSchema=True)

netflix.printSchema()
print(f"netflix rows: {netflix.count()}")
netflix.show(3, truncate=False)

---
## Part 2.1 — Spark equivalents of Part 1 SQL queries

We alias `netflix` as `netflix_df` to keep the query variables tidy.

In [ ]:
netflix_df = netflix

### Q1. Newest 5 movies by release year

In [ ]:
print("Q1: Newest 5 movies by release year")
netflix_df.filter(netflix_df["type"] == "Movie") \
    .orderBy(fc.desc("release_year")) \
    .select("title", "release_year") \
    .show(5)

### Q2. Count of movies per rating (sorted desc)

In [ ]:
print("Q2: Count movies per rating")
netflix_df.filter(netflix_df["type"] == "Movie") \
    .groupBy("rating") \
    .count() \
    .orderBy(fc.desc("count")) \
    .show()

### Q3. Average release year per rating

In [ ]:
print("Q3: Average release year per rating")
netflix_df.filter(netflix_df["type"] == "Movie") \
    .groupBy("rating") \
    .agg(fc.avg("release_year").alias("avg_release_year")) \
    .show()

### Q4. Ratings with more than 3 titles

In [ ]:
print("Q4: Ratings with more than 3 titles (Movies only)")
netflix_df.filter(netflix_df["type"] == "Movie") \
    .groupBy("rating") \
    .count() \
    .filter(fc.col("count") > 3) \
    .show()

### Q5. Uppercase titles, first 10

In [ ]:
print("Q5: Uppercase titles (first 10)")
netflix_df.select(fc.upper(fc.col("title")).alias("uppercase_title")) \
    .show(10, truncate=False)

### Q6. Directors with more than one title

In [ ]:
print("Q6: Directors with more than one title")
netflix_df.filter((netflix_df["type"] == "Movie") & (netflix_df["director"].isNotNull())) \
    .groupBy("director") \
    .count() \
    .filter(fc.col("count") > 1) \
    .show(truncate=False)

### Q7. Titles whose description mentions "father" 

In [ ]:
print("Q7: Titles with 'father' in description")
netflix_df.filter(netflix_df["type"] == "Movie") \
    .filter(fc.lower(fc.col("description")).contains("father")) \
    .select("title") \
    .show(truncate=False)

### Q8. Titles that appear in both tables, year > 1995 (join)

In [ ]:
result_q8 = netflix_df.join(
    movie_df,
    netflix_df["movie_id_np"] == movie_df["movie_id_np"],
    "inner"
).filter(
    movie_df["year"] > 1995
).select(
    netflix_df["title"]
)

print("Q8: Titles in both tables where year > 1995")
result_q8.show(truncate=False)

---
## Part 2.2.1 — Deeper analytics

Standardize `release_year` and filter to movies for the analytics work:

In [ ]:
netflix_ext = netflix
netflix_movies = netflix_ext.filter(fc.col("type") == "Movie") \
    .withColumn("release_year", fc.col("release_year").cast("int"))

### Q1. Average genre breadth by release year

Split `listed_in` on commas and count how many genres each film has. Group by release year to see whether genre tagging has broadened over time. Result: fairly stable at 2–3 genres per movie.

In [ ]:
genre_breadth = netflix_movies.withColumn(
    "genre_count", fc.size(fc.split(fc.col("listed_in"), r",\s*"))
)

avg_breadth_by_year = genre_breadth.groupBy("release_year").agg(
    fc.avg("genre_count").alias("avg_genre_breadth"),
    fc.count("*").alias("movie_count")
).filter(
    fc.col("release_year").isNotNull()
).orderBy("release_year")

avg_breadth_by_year.show(50)

### Q2. Catalog depth per genre

Explode `listed_in` so each (movie, genre) pair is its own row, then compute `newest_year − oldest_year` per genre. Documentaries and Classic Movies top out with 79 and 76 years of depth.

In [ ]:
movies_exploded = netflix_movies.withColumn(
    "genre", fc.explode(fc.split(fc.col("listed_in"), r",\s*"))
)

catalog_depth = movies_exploded.groupBy("genre").agg(
    fc.min("release_year").alias("oldest_year"),
    fc.max("release_year").alias("newest_year"),
    fc.count("*").alias("movie_count")
).withColumn(
    "catalog_depth", fc.col("newest_year") - fc.col("oldest_year")
).orderBy(fc.desc("catalog_depth"))

catalog_depth.show(truncate=False)

### Q3. Rating vs. genre breadth

Filter to known-valid MPAA/TV ratings (dataset is dirty). UR tops out at ~2.67 avg genres, TV-Y bottoms out at ~1.26 — children's content is narrowly categorized.

In [ ]:
valid_ratings = ["G", "PG", "PG-13", "R", "NC-17", "NR", "UR",
                 "TV-Y", "TV-Y7", "TV-G", "TV-PG", "TV-14", "TV-MA"]

rating_genre_breadth = netflix_movies.withColumn(
    "genre_count", fc.size(fc.split(fc.col("listed_in"), r",\s*"))
).filter(
    fc.col("rating").isin(valid_ratings)
).groupBy("rating").agg(
    fc.avg("genre_count").alias("avg_genre_breadth"),
    fc.min("genre_count").alias("min_genres"),
    fc.max("genre_count").alias("max_genres"),
    fc.count("*").alias("movie_count")
).orderBy(fc.desc("avg_genre_breadth"))

rating_genre_breadth.show(truncate=False)

### Q4. Added-year vs release-year lag by genre & decade

Walkthrough of a supplied query. Steps:
1. `netflix_parsed` — parse `date_added` to a date, extract `added_year`
2. `lag_df` — compute `lag = added_year − release_year`
3. `lag_exploded` — one row per (movie, genre); bucket by release decade
4. `lag_summary` — mean, median, p25, p75 of lag per (genre, decade), filtered to groups of ≥5 titles

Older content (1940s–1960s) sits at 50–70+ year lags (added long after theatrical release); recent content collapses to short lags.

In [ ]:
netflix_clean = netflix.filter(fc.col("date_added") != "NaN")

netflix_parsed = netflix_clean.withColumn(
    "date_added_parsed", fc.to_date(fc.trim(fc.col("date_added")), "MMMM d, yyyy")
).withColumn("added_year", fc.year(fc.col("date_added_parsed")))

lag_df = netflix_parsed.withColumn(
    "lag", fc.col("added_year") - fc.col("release_year")
)

lag_exploded = lag_df.withColumn(
    "genre", fc.explode(fc.split(fc.col("listed_in"), r",\s*"))
).withColumn(
    "release_decade", (fc.col("release_year") / 10).cast("int") * 10
)

lag_summary = (
    lag_exploded.groupBy("genre", "release_decade")
    .agg(
        fc.avg("lag").alias("avg_lag"),
        fc.expr("percentile_approx(lag, 0.5)").alias("median_lag"),
        fc.expr("percentile_approx(lag, 0.25)").alias("p25_lag"),
        fc.expr("percentile_approx(lag, 0.75)").alias("p75_lag"),
        fc.count("*").alias("num_titles")
    )
    .filter("num_titles >= 5")
    .orderBy("release_decade", "genre")
)

lag_summary.show(100, truncate=False)

---
## Part 2.2.2 — Query optimization

Take the lag summary from 2.2.1 Q4 as the workload. Measure, then improve.

### Q1. Baseline runtime

In [ ]:
import time

def time_action(df):
    start = time.perf_counter()
    df.count()
    end = time.perf_counter()
    return end - start

baseline_time = time_action(lag_summary)
print(f"Baseline runtime: {baseline_time:.4f} seconds")
# Recorded on Colab: 1.1998 s

### Q2. `.explain(True)` — identify shuffles

Two shuffles show up in the plan:
1. **`Exchange hashpartitioning(genre, release_decade)`** — the `groupBy` needs all rows for a given (genre, decade) key on the same partition.
2. **`Exchange rangepartitioning`** — the final `orderBy("release_decade", "genre")` requires a global sort.

Shuffles are expensive because they push data across the network and to disk.

In [ ]:
lag_summary.explain(True)

### Q3. Cache the exploded intermediate

`lag_exploded` sits right after the row-multiplying `explode` — the most expensive step in the pipeline. Caching it means the (re)aggregation reads from memory instead of recomputing. Runtime dropped from **1.1998 s → 0.9196 s**.

In [ ]:
lag_exploded_cached = lag_exploded.cache()
lag_exploded_cached.count()  # materialize the cache

lag_summary_cached = (
    lag_exploded_cached.groupBy("genre", "release_decade")
    .agg(
        fc.avg("lag").alias("avg_lag"),
        fc.expr("percentile_approx(lag, 0.5)").alias("median_lag"),
        fc.expr("percentile_approx(lag, 0.25)").alias("p25_lag"),
        fc.expr("percentile_approx(lag, 0.75)").alias("p75_lag"),
        fc.count("*").alias("num_titles")
    )
    .filter("num_titles >= 5")
    .orderBy("release_decade", "genre")
)

cached_time = time_action(lag_summary_cached)
print(f"Cached runtime:  {cached_time:.4f} seconds")
print(f"Baseline was:    {baseline_time:.4f} seconds")

### Q4. Column pruning + push filters earlier

Only `date_added`, `release_year`, `listed_in` are needed downstream. Selecting those columns early cuts memory + I/O. Runtime: **1.0483 s** (vs 1.1998 s baseline).

In [ ]:
netflix_pruned = netflix.filter(
    fc.col("date_added") != "NaN"
).select("date_added", "release_year", "listed_in")

netflix_parsed_pruned = netflix_pruned.withColumn(
    "added_year", fc.year(fc.to_date(fc.trim(fc.col("date_added")), "MMMM d, yyyy"))
).withColumn(
    "lag", fc.col("added_year") - fc.col("release_year")
).select("lag", "release_year", "listed_in")

lag_exploded_pruned = netflix_parsed_pruned.withColumn(
    "genre", fc.explode(fc.split(fc.col("listed_in"), r",\s*"))
).withColumn(
    "release_decade", (fc.col("release_year") / 10).cast("int") * 10
).select("genre", "release_decade", "lag")

lag_summary_pruned = (
    lag_exploded_pruned.groupBy("genre", "release_decade")
    .agg(
        fc.avg("lag").alias("avg_lag"),
        fc.expr("percentile_approx(lag, 0.5)").alias("median_lag"),
        fc.expr("percentile_approx(lag, 0.25)").alias("p25_lag"),
        fc.expr("percentile_approx(lag, 0.75)").alias("p75_lag"),
        fc.count("*").alias("num_titles")
    )
    .filter("num_titles >= 5")
    .orderBy("release_decade", "genre")
)

pruned_time = time_action(lag_summary_pruned)
print(f"Pruned runtime:  {pruned_time:.4f} seconds")
print(f"Baseline was:    {baseline_time:.4f} seconds")

### Q5. Extra optimization — repartition by groupBy keys

Repartition upstream by (`genre`, `release_decade`) so the downstream `groupBy` sees pre-aligned data. On a tiny single-node Colab cluster the improvement is data-distribution dependent; the technique is what matters.

In [ ]:
lag_exploded_repartitioned = lag_exploded_pruned.repartition(4, "genre", "release_decade")

lag_summary_repartitioned = (
    lag_exploded_repartitioned.groupBy("genre", "release_decade")
    .agg(
        fc.avg("lag").alias("avg_lag"),
        fc.expr("percentile_approx(lag, 0.5)").alias("median_lag"),
        fc.expr("percentile_approx(lag, 0.25)").alias("p25_lag"),
        fc.expr("percentile_approx(lag, 0.75)").alias("p75_lag"),
        fc.count("*").alias("num_titles")
    )
    .filter("num_titles >= 5")
    .orderBy("release_decade", "genre")
)

repartition_time = time_action(lag_summary_repartitioned)
print(f"Repartitioned runtime: {repartition_time:.4f} seconds")
print(f"Baseline was:          {baseline_time:.4f} seconds")